## Next Steps

After pre-training or continued pre-training, you typically want to:

- **Supervised Fine-Tuning (SFT)**: Teach the model to follow instructions and perform specific tasks. See `02_finetuning.ipynb` for examples.
- **Alignment**: Use RLHF, DPO, or GRPO to align the model with human preferences. See `04_alignment.ipynb` for alignment recipes.
- **Evaluation**: Benchmark your model on standard tasks to measure the impact of domain-specific pre-training.

Pre-training creates the foundation, but fine-tuning and alignment make the model truly useful for downstream applications.

In [ ]:
config = TrainConfig(
    recipe="pretrain",
    model=ModelConfig(name="meta-llama/Llama-3.1-8B"),
    data=DataConfig(path="data/corpus/", format="text", streaming=True),
    trainer=TrainerConfig(
        strategy="fsdp",
        mixed_precision="bf16",
        batch_size=2,
        gradient_accumulation=16,
        learning_rate=3e-4,
        max_steps=100000,
    ),
)

state = xaytune.pretrain(config=config)

## Multi-GPU Training with FSDP

For large models and datasets, distribute training across multiple GPUs using Fully Sharded Data Parallel (FSDP).

FSDP shards the model parameters, gradients, and optimizer states across GPUs, allowing you to train models that wouldn't fit on a single GPU. Combined with mixed precision training (bf16), this enables efficient large-scale pre-training.

In [ ]:
config = TrainConfig(
    recipe="pretrain",
    model=ModelConfig(name="meta-llama/Llama-3.1-8B"),
    data=DataConfig(
        path="data/long_documents/",
        format="text",
        max_seq_length=8192,
        packing=True,
    ),
    trainer=TrainerConfig(
        strategy="fsdp",
        batch_size=1,
        gradient_accumulation=32,
        learning_rate=2e-4,
        max_steps=50000,
    ),
)

state = xaytune.pretrain(config=config)

## Long-Context Training

Modern models can handle increasingly long context windows (8K, 16K, 32K tokens or more). Training with longer sequences allows the model to capture longer-range dependencies.

Note that longer sequences require more memory and slower training. Reduce batch size and increase gradient accumulation to fit in GPU memory.

In [ ]:
from xaytune.data.packing import pack_sequences

# Example of how packing works under the hood
sequences = [
    {"input_ids": [1, 2, 3], "attention_mask": [1, 1, 1]},
    {"input_ids": [4, 5], "attention_mask": [1, 1]},
    {"input_ids": [6, 7, 8, 9], "attention_mask": [1, 1, 1, 1]},
]

packed = pack_sequences(sequences, max_seq_length=8, pad_token_id=0)
# Packs [1,2,3] + [4,5] into one sequence, [6,7,8,9] into another
# Result: [[1,2,3,4,5,0,0,0], [6,7,8,9,0,0,0,0]]

## Sequence Packing

Sequence packing concatenates multiple short sequences to fill the maximum sequence length, dramatically improving GPU utilization.

Without packing, a batch of short sequences wastes computation on padding tokens. With packing, every token in the batch is a real token from your corpus.

Here's how packing works under the hood:

In [ ]:
from xaytune.config.schema import DataConfig, ModelConfig, TrainConfig, TrainerConfig

config = TrainConfig(
    recipe="pretrain",
    model=ModelConfig(name="meta-llama/Llama-3.1-8B"),
    data=DataConfig(
        path="data/large_corpus/",
        format="text",
        streaming=True,
        max_seq_length=4096,
        packing=True,
    ),
    trainer=TrainerConfig(
        strategy="fsdp",
        batch_size=2,
        gradient_accumulation=16,
        learning_rate=3e-4,
        max_steps=100000,
        warmup_steps=2000,
    ),
)

state = xaytune.pretrain(config=config)

## Streaming Large Datasets

For very large corpora that don't fit in memory, use streaming mode. This processes data on-the-fly without loading the entire dataset at once.

Streaming is essential for web-scale pre-training where your corpus might be terabytes of text.

In [ ]:
state = xaytune.pretrain(
    model="meta-llama/Llama-3.1-8B",
    dataset="data/medical_papers/",
    format="text",
    learning_rate=3e-4,
    batch_size=2,
    num_epochs=1,
)

## Continued Pre-Training on Domain Data

Continued pre-training adapts a pre-trained model to a specialized domain. This is useful when you have domain-specific text (medical papers, legal documents, code repositories) and want to improve the model's performance on that domain without starting from scratch.

The model retains its general language understanding while learning domain-specific vocabulary, patterns, and knowledge.

In [ ]:
import xaytune

state = xaytune.pretrain(
    model="meta-llama/Llama-3.1-8B",
    dataset="data/corpus/",
    format="text",
)

## Basic Pre-Training

The simplest way to start pre-training is with a one-liner. Point xaytune at your text corpus and specify the model.

# Pre-Training with xaytune

This notebook demonstrates how to pre-train language models from scratch or perform continued pre-training on domain-specific text corpora using xaytune.

**Pre-training** teaches a model to predict the next token in a sequence by training on large amounts of raw text. This is how foundation models like GPT and Llama are initially trained.

**Continued pre-training** adapts an existing pre-trained model to a new domain (medical, legal, code, etc.) by further training it on domain-specific text, improving performance on specialized tasks without losing general capabilities.